In [ ]:
import pandas as pd
import torch
import triton

In [ ]:
# nn.LSTM and Graph sometimes go oom -- drop all nans for this run to avoid misleading summary stats in describe
df1 = (
    pd.read_parquet("../full_fp32.parquet")
    .dropna()
    .reset_index()
    .query("level_0!='s5'")
)
df2 = pd.read_parquet("../full_bf16.parquet").dropna()
df3 = (
    pd.read_parquet("../full_fp16.parquet")
    .dropna()
    .reset_index()
    .query("level_0!='s5'")
)

display(df1.describe())
display(df2.describe())
display(df3.describe())

In [ ]:
# forward runtime visualisation
df_in = pd.read_parquet("../fwd_bf16.parquet").dropna()
dfr = df_in / df_in["lstm"].to_numpy().reshape(-1, 1)

df_fwd = dfr["fast"].droplevel(-1).unstack(-1)

df_fwd.columns = pd.MultiIndex.from_tuples(
    [
        (
            f"FWD_bf16: FastLSTM/nn.LSTM -- torch:{torch.__version__} -- triton:{triton.__version__} -- RTX 2000 Ada",
            f"hidden:{64 << int(c[1:])}",
        )
        for c in df_fwd
    ]
)

df_fwd.index = pd.MultiIndex.from_tuples(
    [(f"seq:{64 << int(s[1:])}", f"batch:{4 << int(b[1:])}") for s, b in df_fwd.index]
)

df_fwd = df_fwd.style.background_gradient(cmap="bwr", vmin=1 - 1, vmax=1 + 1).format(
    "{:.2f}"
)

df_fwd

In [ ]:
from torch.amp import autocast
from fastlstm.lstm import FastLSTM, PersistentLSTMfn
import torch
import torch.nn as nn

from triton.testing import do_bench

hidden_size = 256
x = torch.randn((1, 8, hidden_size), device="cuda")


if False:
    m = nn.LSTM(
        input_size=hidden_size,
        hidden_size=hidden_size,
        device="cuda",
        #    version="persistent",
    )
else:
    m = FastLSTM(
        input_size=hidden_size,
        hidden_size=hidden_size,
        device="cuda",
        version="persistent",
    )


def fn(dtype):
    dd = {"fp32": None, "fp16": torch.float16, "bf16": torch.bfloat16}
    with autocast("cuda", dd[dtype], enabled=dtype != "fp32"):
        m(x)[0].sum().backward()


y = x + 12
with autocast("cuda", torch.float16):
    print(y.dtype)
    t = m(y)[0]
    print(t.dtype)
    t.sum().backward()
    for n, p in m.named_parameters():
        print(n, p.dtype, p.grad.dtype)
    print(y.dtype)

In [ ]:
import pandas as pd

ax = (
    pd.DataFrame(
        res,
        columns=["batch_size", "nn.LSTM", "persistent", "graph", "cuda_fused", "cuda"],
    )
    .set_index("batch_size")[
        ["nn.LSTM", "graph", "persistent", "Flash:alternating", "Flash:fused"]
    ]
    .plot(
        kind="bar",
        ylabel="mean runtime [ms]",
        title="fp16 FWD pass on H100 -- seq=1024, hidden=768",
        grid=True,
    )
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f")

ax

In [ ]:
# visualise overfit losses
import pandas as pd

df = pd.read_parquet("../bf16_losses.parquet")
df.plot(logy=True)

In [ ]:
from collections import defaultdict
import json
import re

import numpy as np
import pandas as pd

with open("../tuning_full_fp16.json") as fh:
    data = json.load(fh)

res = defaultdict(list)

for setting, configs in data.items():
    for config, perf in configs:
        if setting.startswith("persistent"):
            try:
                matches = re.findall(
                    "BLOCK_SIZE_K: ([0-9]+), batch_chunks: [0-9]+, num_pid_b: [0-9]+, BLOCK_SIZE_B: ([0-9]+), BLOCK_SIZE_H: ([0-9]+), num_warps: ([0-9]+), num_ctas: 1, num_stages: ([0-9]+), maxnreg: None",
                    config,
                )
                assert len(matches) == 1
                k, h, b, w, s = (int(e) for e in matches[0])
                res[(setting, h, b, k, w, s)] += [perf[0]]

            except:
                matches = re.findall(
                    "BLOCK_SIZE_K: ([0-9]+), BLOCK_SIZE_H: ([0-9]+), BLOCK_SIZE_B: ([0-9]+), num_pid_h: ([0-9]+), num_pid_b: ([0-9]+), num_warps: ([0-9]+), num_ctas: 1, num_stages: ([0-9]+), maxnreg: None",
                    config,
                )
                assert len(matches) == 1
                k, h, b, nh, nb, w, s = (int(e) for e in matches[0])

                res[(setting, h, b, k, nh, nb, w, s)] += [perf[0]]

        else:
            matches = re.findall(
                "BLOCK_SIZE_H: ([0-9]+), BLOCK_SIZE_B: ([0-9]+), BLOCK_SIZE_K: ([0-9]+), GROUP_SIZE_B: ([0-9]+), num_warps: ([0-9]+), num_ctas: 1, num_stages: ([0-9]+), maxnreg: None",
                config,
            )
            assert len(matches) == 1
            h, b, k, g, w, s = (int(e) for e in matches[0])

            res[(setting, h, b, k, g, w, s)] += [perf[0]]

for k, v in res.items():
    res[k] = np.array(v).mean()


raw = pd.DataFrame(res, index=["val"]).stack(0).T.droplevel(0, axis=1)


best = pd.concat([raw.idxmin(), 1_000 * raw.min()], axis=1, keys=["config", "time"])
best.index = pd.MultiIndex.from_tuples([tuple(ii.split("-")) for ii in best.index])

best = best.droplevel(-1)

In [ ]:
df_in = pd.read_parquet("../full_fp16.parquet")
# df_in = pd.read_parquet("../figures/raw_data/fwd_fp16.parquet")
df_in = pd.read_parquet("../figures/raw_data/full_fp16.parquet")


df_in["fast"] = df_in[["persistent", "graph"]].min(axis=1)
# df_in["flash"] = df_in[["cuda", "cuda_fused", "triton_fused"]].min(axis=1)
dfr = df_in / df_in["lstm"].to_numpy().reshape(-1, 1)
dfr["lstm"] = df_in["lstm"]

dfr.index = pd.MultiIndex.from_tuples(
    [
        (int(ii[0][1:]), int(ii[1][1:]), int(ii[2][1:]), int(ii[3][1:]))
        for ii in dfr.index
    ],
    names=["seq", "batch", "hidden", "layer"],
)

rn = {
    "fast": "fast/lstm",
    #   "flash": "flash/lstm",
    "lstm": "lstm[ms]",
    "graph": "graph/lstm",
    "persistent": "persistent/lstm",
}

(
    dfr[
        [
            "graph",
            "persistent",
            "fast",
            #    "flash", "lstm"
        ]
    ]
    .rename(columns=rn)
    .droplevel(-1, axis=0)
    .unstack(-1)
    .style.background_gradient(
        cmap="bwr",
        vmin=1 - 1,
        vmax=1 + 1,
        subset=[v for v in rn.values() if not v.startswith("lstm")],
    )
    .format("{:.2f}")
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, ax = plt.subplots(ncols=2, figsize=(12, 4))

for i in range(2):
    name = ["FWD", "FWD+BWD"][i]
    n2 = ["fwd", "full"][i]

    df = pd.read_parquet(f"../figures/raw_data/{n2}_fig4_h100_fp16.parquet").rename(columns={"persistent": "fast"})
    df = df.droplevel([0, 2, 3])

    df.index = [int(i[1:]) for i in df.index]

    df.sort_index()[["lstm", "fast", "cuda", "cuda_fused"]].plot(
        kind="bar",
        ylabel="mean runtime [ms]",
        title=name,
        grid=True,
        ax=ax[i]
    )

    for container in ax[i].containers:
        ax[i].bar_label(container, fmt="%.1f")


fig.suptitle(f"LSTM in fp16 on H100 SXM -- seq=1024, hidden=768 -- torch:{torch.__version__} - triton:{triton.__version__}")
fig.tight_layout()

# fig.savefig("../figures/flashrnn_fig4.png", dpi=300, bbox_inches="tight")

